# Acoustic defect classification
Baseline workflow for extracting audio features and evaluating a Random Forest classifier. Raw WAV files and labels must be supplied under `part1_inspection/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

DATA = Path('part1_inspection')
AUDIO = DATA / 'train' / 'audio'
LABELS = pd.read_csv(DATA / 'train' / 'labels.csv')


In [ ]:
def extract_audio_features(path):
    signal, sr = librosa.load(path, sr=None, mono=True)
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=13)
    rms = librosa.feature.rms(y=signal)
    zcr = librosa.feature.zero_crossing_rate(signal)
    centroid = librosa.feature.spectral_centroid(y=signal, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=signal, sr=sr)
    row = {'rms_mean': float(rms.mean()), 'rms_std': float(rms.std()),
           'zcr_mean': float(zcr.mean()), 'centroid_mean': float(centroid.mean()),
           'bandwidth_mean': float(bandwidth.mean())}
    for i in range(13):
        row[f'mfcc_{i+1}_mean'] = float(mfcc[i].mean())
        row[f'mfcc_{i+1}_std'] = float(mfcc[i].std())
    return row

rows = []
for unit_id in LABELS.unit_id.astype(str):
    path = AUDIO / f'{unit_id}.wav'
    if path.exists():
        rows.append({'unit_id': unit_id, **extract_audio_features(path)})
features = pd.DataFrame(rows)
data = LABELS.assign(unit_id=LABELS.unit_id.astype(str)).merge(features, on='unit_id', validate='one_to_one')
feature_cols = [c for c in features.columns if c != 'unit_id']
X = data[feature_cols]
y = data.acoustic_label.astype(str).str.upper().eq('NG').astype(int)
print('Feature matrix:', X.shape, 'Class counts:', y.value_counts().to_dict())


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                               min_samples_leaf=2, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
scores = model.predict_proba(X_val)[:, 1]
pred = (scores >= 0.24).astype(int)
print(confusion_matrix(y_val, pred, labels=[0, 1]))
print(classification_report(y_val, pred, target_names=['OK', 'NG'], zero_division=0))
print('ROC-AUC:', roc_auc_score(y_val, scores))
